# 03 - Similarity Graph Construction

This notebook constructs the farm similarity graph used before spectral clustering. Farms are nodes, and edges connect farms that are close in the PCA-reduced feature space.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.graph_construction import run_graph_construction

PROCESSED_DIR = PROJECT_ROOT / "outputs" / "processed"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"

## Build the Graph

We use the first 6 principal components because Step 5 showed they retain about 95% of the variance. Edges use Gaussian similarity based on Euclidean distance in this PCA space.

A neighbor sensitivity check showed that smaller k-nearest-neighbor graphs were split into many disconnected components. The saved graph uses `k=400`, which creates one connected graph across all farms.

In [2]:
graph_result = run_graph_construction(
    pca_coordinates_path=PROCESSED_DIR / "pca_coordinates_all_components.csv",
    pca_2d_path=PROCESSED_DIR / "pca_coordinates_2d.csv",
    output_dir=PROCESSED_DIR,
    table_dir=TABLE_DIR,
    figure_dir=FIGURE_DIR,
    n_components=6,
    n_neighbors=400,
)

graph_result.metadata

{'nodes': 2789,
 'edges': 632680,
 'requested_neighbors': 400,
 'pca_components_used': 6,
 'sigma': 2.4661658846005134,
 'connected_components': 1,
 'largest_component_size': 2789,
 'largest_component_pct': 1.0,
 'min_degree': 400,
 'median_degree': 440.0,
 'mean_degree': 453.6966654714952,
 'max_degree': 652,
 'min_similarity': 0.005479023582268659,
 'median_similarity': 0.5954884875163858,
 'mean_similarity': 0.46269494842424186,
 'max_similarity': 1.0}

## Neighbor Sensitivity

This table shows how graph connectivity changes as the number of nearest neighbors increases.

In [3]:
sensitivity = pd.read_csv(TABLE_DIR / "similarity_graph_neighbor_sensitivity.csv")
sensitivity

,n_neighbors,edges,connected_components,largest_component_size,largest_component_pct,mean_degree,median_degree,sigma
0,10,16787,147,42,0.0151,12.04,10.0,0.0191
1,15,24233,104,70,0.0251,17.38,15.0,0.0191
2,20,32033,80,90,0.0323,22.97,20.0,0.0191
3,30,47014,45,137,0.0491,33.71,30.0,0.0273
4,40,61502,40,137,0.0491,44.10,40.0,0.0299
5,50,74175,38,149,0.0534,53.19,50.0,0.0435
6,75,122670,16,267,0.0957,87.97,78.0,0.0716
7,100,165486,9,450,0.1613,118.67,113.0,0.1039
8,150,243072,6,605,0.2169,175.11,168.0,0.1762
9,200,322545,2,1853,0.6644,231.98,231.0,2.2089


## Saved Outputs

In [4]:
for path in [
    PROCESSED_DIR / "similarity_graph_edges.csv",
    TABLE_DIR / "similarity_graph_degrees.csv",
    TABLE_DIR / "similarity_graph_metadata.json",
    TABLE_DIR / "similarity_graph_neighbor_sensitivity.csv",
    FIGURE_DIR / "similarity_graph_degree_distribution.png",
    FIGURE_DIR / "similarity_graph_weight_distribution.png",
    FIGURE_DIR / "similarity_graph_pca_overlay.png",
]:
    print(path.relative_to(PROJECT_ROOT))

outputs/processed/similarity_graph_edges.csv
outputs/tables/similarity_graph_degrees.csv
outputs/tables/similarity_graph_metadata.json
outputs/tables/similarity_graph_neighbor_sensitivity.csv
outputs/figures/similarity_graph_degree_distribution.png
outputs/figures/similarity_graph_weight_distribution.png
outputs/figures/similarity_graph_pca_overlay.png
